# Module 03 — Unsupervised & Statistical Learning
## 03-09: Matrix Factorization & Decomposition

**Objective:** Implement SVD, truncated SVD, matrix completion, and Non-negative Matrix Factorization (NMF) from scratch, then apply them to dimensionality reduction, parts-based representation, and low-rank approximation.

**Prerequisites:** 1-06 (Linear Algebra — SVD foundations)

## Part 0 — Setup & Prerequisites

This notebook owns **SVD-based dimensionality reduction**, **truncated SVD**, **Non-negative Matrix Factorization (NMF)**, and **matrix completion**. The linear algebra foundations (eigendecomposition, SVD definition) were taught in 1-06 and are referenced here without re-derivation. These techniques connect forward to recommender systems (Module 19) where user–item matrices are factored, and to LoRA (Module 13) where low-rank factorization enables efficient fine-tuning of LLMs.

**Prerequisites:** 1-06 (Linear Algebra)

In [ ]:
import sys
import time
import warnings
import random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import load_digits
from sklearn.decomposition import TruncatedSVD as SklearnTSVD
from sklearn.decomposition import NMF as SklearnNMF
from sklearn.preprocessing import MinMaxScaler
import torch

print(f'Python : {sys.version.split()[0]}')
print(f'NumPy  : {np.__version__}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Experiment constants ──────────────────────────────────────────────────────
N_COMPONENTS_SVD  = 20    # Truncated SVD components for Digits
N_COMPONENTS_NMF  = 16    # NMF components for Digits (4×4 grid)
NMF_MAX_ITER      = 300   # Maximum NMF multiplicative update iterations
NMF_TOL           = 1e-4  # NMF convergence tolerance
NMF_EPS           = 1e-10 # NMF numerical floor
# Synthetic matrix for matrix completion
M_ROWS            = 80    # Rows of synthetic observation matrix
N_COLS            = 60    # Cols of synthetic observation matrix
TRUE_RANK         = 5     # True rank of the synthetic matrix
MISSING_FRAC      = 0.30  # Fraction of entries to mask as missing
# LoRA demo
LORA_RANK         = 8     # Low-rank dimension for LoRA demo
WEIGHT_DIM_IN     = 128
WEIGHT_DIM_OUT    = 64

print('Configuration loaded.')
print(f'  N_COMPONENTS_SVD = {N_COMPONENTS_SVD}')
print(f'  N_COMPONENTS_NMF = {N_COMPONENTS_NMF}')
print(f'  Synthetic matrix : {M_ROWS} x {N_COLS}, rank {TRUE_RANK}, missing {MISSING_FRAC:.0%}')

### Data Loading & EDA

We use two datasets:
- **sklearn Digits:** 1,797 hand-written digit images (8×8 = 64 pixels). Each row is one image. We treat this as a tall data matrix and apply SVD and NMF to find latent structure.
- **Synthetic low-rank matrix:** A rank-5 matrix with added noise and random missingness, used to demonstrate matrix completion.

In [ ]:
# ── Load sklearn Digits ───────────────────────────────────────────────────────
digits = load_digits()
X_digits: np.ndarray = digits.data.astype(np.float64)  # (1797, 64)
y_digits: np.ndarray = digits.target

print('Digits dataset')
print(f'  X shape  : {X_digits.shape}  (samples x pixels)')
print(f'  y shape  : {y_digits.shape}')
print(f'  Classes  : {np.unique(y_digits)}')
print(f'  Pixel range: [{X_digits.min():.1f}, {X_digits.max():.1f}]')
print(f'  Class distribution: {np.bincount(y_digits)}')

# ── Normalise pixels to [0, 1] for NMF (requires non-negative input) ──────────
scaler_digits = MinMaxScaler()
X_digits_nn: np.ndarray = scaler_digits.fit_transform(X_digits)  # non-negative, for NMF

# ── Visualise sample images ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for digit_class in range(10):
    idx_list = np.where(y_digits == digit_class)[0]
    for row_offset in range(2):
        img = X_digits[idx_list[row_offset]].reshape(8, 8)
        axes[row_offset, digit_class].imshow(img, cmap='gray_r')
        axes[row_offset, digit_class].axis('off')
        if row_offset == 0:
            axes[row_offset, digit_class].set_title(str(digit_class), fontsize=9)
plt.suptitle('Sample Images from Digits Dataset (2 per class)', y=1.02)
plt.tight_layout()
plt.savefig('digits_samples.png', dpi=100, bbox_inches='tight')
plt.show()

# ── Synthetic low-rank matrix for matrix completion ───────────────────────────
rng_synth = np.random.RandomState(SEED)
U_true = rng_synth.randn(M_ROWS, TRUE_RANK)
V_true = rng_synth.randn(N_COLS, TRUE_RANK)
X_true: np.ndarray = U_true @ V_true.T + 0.1 * rng_synth.randn(M_ROWS, N_COLS)

# Create observed matrix with random missing entries
mask_obs: np.ndarray = rng_synth.rand(M_ROWS, N_COLS) > MISSING_FRAC  # True = observed
X_observed: np.ndarray = X_true.copy()
X_observed[~mask_obs] = np.nan

print(f'\nSynthetic matrix: {M_ROWS}x{N_COLS}, true rank={TRUE_RANK}')
print(f'  Observed entries : {mask_obs.sum()} / {M_ROWS * N_COLS} ({mask_obs.mean():.1%})')
print(f'  Missing entries  : {(~mask_obs).sum()} ({(~mask_obs).mean():.1%})')

---
## Part 1 — SVD & NMF from Scratch

### 1.1 Singular Value Decomposition (SVD)

Every matrix $X \in \mathbb{R}^{m \times n}$ admits the decomposition:

$$X = U \Sigma V^\top$$

where $U \in \mathbb{R}^{m \times r}$ (left singular vectors, orthonormal columns), $\Sigma \in \mathbb{R}^{r \times r}$ (diagonal, $\sigma_1 \geq \sigma_2 \geq \dots \geq 0$), and $V \in \mathbb{R}^{n \times r}$ (right singular vectors, orthonormal columns), with $r = \min(m, n)$.

**Connection to eigendecomposition (see 1-06):** $X^\top X = V \Sigma^2 V^\top$ — the right singular vectors are eigenvectors of $X^\top X$, and the singular values are square roots of eigenvalues.

**Eckart–Young theorem:** The best rank-$k$ approximation to $X$ in Frobenius norm is:
$$\hat{X}_k = U_k \Sigma_k V_k^\top, \quad \|X - \hat{X}_k\|_F^2 = \sum_{i=k+1}^{r} \sigma_i^2$$

### 1.2 Non-negative Matrix Factorization (NMF)

NMF (Lee & Seung, 1999) constrains $W, H \geq 0$:
$$X \approx W H, \quad W \in \mathbb{R}^{m \times k}_{\geq 0},\; H \in \mathbb{R}^{k \times n}_{\geq 0}$$

The non-negativity constraint yields **parts-based** representations: each column of $W$ is a non-negative basis component (e.g., stroke of a digit), and each column of $H$ is a non-negative combination of those parts.

**Lee–Seung multiplicative update rules** (Frobenius objective):
$$H \leftarrow H \odot \frac{W^\top X}{W^\top W H + \varepsilon}, \quad W \leftarrow W \odot \frac{X H^\top}{W H H^\top + \varepsilon}$$

In [ ]:
# ── SVD via eigendecomposition of X^T X ──────────────────────────────────────

def compute_svd_eig(X: np.ndarray) -> tuple:
    '''Compute SVD of X using eigendecomposition of X^T X.

    Implements X = U Sigma V^T without calling np.linalg.svd.
    Steps:
      1. Eigendecompose X^T X = V Lambda V^T  (eigh returns ascending order).
      2. Singular values s_i = sqrt(lambda_i), flip to descending.
      3. Left singular vectors: U = X V / s.

    Args:
        X: Input matrix of shape (m, n), float64 recommended.

    Returns:
        Tuple (U, s, Vt) with shapes (m, r), (r,), (r, n) where r = min(m, n).
    '''
    XtX: np.ndarray = X.T @ X                       # (n, n)
    eigenvalues, V = np.linalg.eigh(XtX)             # ascending eigenvalues
    # Flip to descending order
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    V = V[:, idx]                                     # (n, r)
    # Clip small negative eigenvalues caused by floating-point errors
    eigenvalues = np.maximum(eigenvalues, 0.0)
    s: np.ndarray = np.sqrt(eigenvalues)              # singular values (n,)
    # Keep only non-trivially non-zero singular values
    r: int = int(np.sum(s > 1e-10))
    V_r = V[:, :r]                                   # (n, r)
    s_r = s[:r]                                      # (r,)
    U_r: np.ndarray = X @ V_r / s_r[np.newaxis, :]  # (m, r)
    return U_r, s_r, V_r.T                           # Vt: (r, n)


def truncated_svd_manual(X: np.ndarray, k: int) -> tuple:
    '''Return top-k singular components of X.

    Args:
        X: Matrix of shape (m, n).
        k: Number of singular values/vectors to retain.

    Returns:
        Tuple (U_k, s_k, Vt_k) with shapes (m, k), (k,), (k, n).
    '''
    U, s, Vt = compute_svd_eig(X)
    k_actual: int = min(k, len(s))
    return U[:, :k_actual], s[:k_actual], Vt[:k_actual, :]


def reconstruct_from_svd(U: np.ndarray, s: np.ndarray, Vt: np.ndarray) -> np.ndarray:
    '''Reconstruct matrix from SVD components: X_hat = U diag(s) Vt.

    Args:
        U: Left singular vectors (m, k).
        s: Singular values (k,).
        Vt: Right singular vectors transposed (k, n).

    Returns:
        Reconstructed matrix of shape (m, n).
    '''
    return (U * s[np.newaxis, :]) @ Vt


def frobenius_error(X: np.ndarray, X_hat: np.ndarray) -> float:
    '''Relative Frobenius reconstruction error ||X - X_hat||_F / ||X||_F.

    Args:
        X: Ground-truth matrix.
        X_hat: Approximation matrix.

    Returns:
        Scalar relative error.
    '''
    return float(np.linalg.norm(X - X_hat, 'fro') / (np.linalg.norm(X, 'fro') + 1e-10))


def explained_variance_ratio_svd(s: np.ndarray) -> np.ndarray:
    '''Compute cumulative explained variance ratio from singular values.

    Args:
        s: Singular values in descending order, shape (r,).

    Returns:
        Cumulative explained variance ratios, shape (r,).
    '''
    s2: np.ndarray = s ** 2
    return np.cumsum(s2) / (np.sum(s2) + 1e-10)


# ── Quick verification ─────────────────────────────────────────────────────────
X_toy = np.array([[3., 2., 1.], [2., 5., 3.], [1., 3., 4.]], dtype=np.float64)
U_t, s_t, Vt_t = compute_svd_eig(X_toy)
X_rec_t = reconstruct_from_svd(U_t, s_t, Vt_t)
ref_s = np.linalg.svd(X_toy, compute_uv=False)

print('SVD verification on 3x3 toy matrix:')
print(f'  Our singular values  : {s_t.round(4)}')
print(f'  NumPy reference      : {ref_s.round(4)}')
print(f'  Max singular value diff: {np.max(np.abs(s_t - ref_s)):.2e}')
print(f'  Reconstruction error : {frobenius_error(X_toy, X_rec_t):.2e}')

### Randomised SVD via Power Iteration

The eigendecomposition approach computes all $n$ eigenvalues of $X^T X$ even when we only want the top $k$. For large matrices, **randomised SVD** (Halko et al., 2011) is far more efficient: it projects $X$ onto a random subspace, then performs a small exact SVD on the much smaller projected matrix. Power iterations improve the approximation quality.

In [ ]:
import time as _time

def compute_svd_power_iteration(
    X: np.ndarray,
    k: int,
    n_power_iter: int = 4,
    oversampling: int = 10,
    random_state: int = SEED,
) -> tuple:
    '''Randomised SVD via power iteration (Halko, Martinsson & Tropp, 2011).

    Algorithm:
      1. Draw a random Gaussian matrix Omega of size (n, k + oversampling).
      2. Form Y = (X X^T)^q X Omega  (q = n_power_iter) to sharpen the approximation.
      3. Compute thin QR: Y = Q R  (orthonormal basis for range of Y).
      4. Project X into Q-space: B = Q^T X.
      5. Compute small SVD of B; recover U = Q U_B.

    Args:
        X: Input matrix of shape (m, n).
        k: Number of singular components to keep.
        n_power_iter: Number of power iterations (higher => better approximation).
        oversampling: Extra columns to improve approximation quality.
        random_state: RNG seed.

    Returns:
        Tuple (U_k, s_k, Vt_k) with shapes (m, k), (k,), (k, n).
    '''
    rng = np.random.RandomState(random_state)
    m, n = X.shape
    l: int = k + oversampling                         # oversampled dimension
    Omega: np.ndarray = rng.randn(n, l)               # random projection
    Y: np.ndarray = X @ Omega                         # (m, l)
    # Power iterations: Y <- (X X^T)^q Y
    for _ in range(n_power_iter):
        Y = X @ (X.T @ Y)                             # two matrix-vector products
    Q, _ = np.linalg.qr(Y)                           # orthonormal basis (m, l)
    B: np.ndarray = Q.T @ X                          # small matrix (l, n)
    U_b, s_b, Vt_b = np.linalg.svd(B, full_matrices=False)
    U_k: np.ndarray = Q @ U_b[:, :k]                 # recover left singular vectors
    return U_k, s_b[:k], Vt_b[:k, :]


# ── Benchmark: eigendecomp vs power iteration vs numpy linalg ─────────────────
print('=== SVD Algorithm Comparison ===\n')
print(f'Dataset: {X_digits.shape}, extracting top {N_COMPONENTS_SVD} components\n')

methods: list = [
    ('Eigendecomp (ours)',       lambda: truncated_svd_manual(X_digits, N_COMPONENTS_SVD)),
    ('Power iteration (ours)',   lambda: compute_svd_power_iteration(X_digits, N_COMPONENTS_SVD, n_power_iter=4)),
    ('NumPy linalg.svd (full)', lambda: np.linalg.svd(X_digits, full_matrices=False)),
]

timing_results: list = []
for method_name, fn in methods:
    t0 = _time.perf_counter()
    result_svd = fn()
    elapsed = _time.perf_counter() - t0
    if isinstance(result_svd, tuple) and len(result_svd) == 3:
        U_m, s_m, Vt_m = result_svd
        if len(s_m) > N_COMPONENTS_SVD:
            U_m = U_m[:, :N_COMPONENTS_SVD]
            s_m = s_m[:N_COMPONENTS_SVD]
            Vt_m = Vt_m[:N_COMPONENTS_SVD, :]
        X_rec_m = reconstruct_from_svd(U_m, s_m, Vt_m)
        err_m = frobenius_error(X_digits, X_rec_m)
    else:
        err_m = float('nan')
    timing_results.append({'Method': method_name, 'Time (s)': elapsed, 'Rel Error': err_m})
    print(f'  {method_name:<30} | time={elapsed:.4f}s | rel_err={err_m:.4f}')

# Compare singular values: eigendecomp vs power iteration
U_eig_cmp, s_eig_cmp, _ = truncated_svd_manual(X_digits, N_COMPONENTS_SVD)
U_pi_cmp, s_pi_cmp, _ = compute_svd_power_iteration(X_digits, N_COMPONENTS_SVD)
sv_corr: float = float(np.corrcoef(s_eig_cmp, s_pi_cmp)[0, 1])
sv_max_diff: float = float(np.max(np.abs(s_eig_cmp - s_pi_cmp)))
print(f'\nSingular value comparison (eigendecomp vs power iter):')
print(f'  Pearson correlation: {sv_corr:.6f}')
print(f'  Max absolute diff  : {sv_max_diff:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.plot(np.arange(1, N_COMPONENTS_SVD + 1), s_eig_cmp, 'b-o', ms=4, label='Eigendecomp')
ax.plot(np.arange(1, N_COMPONENTS_SVD + 1), s_pi_cmp, 'r--s', ms=4, label='Power iter (q=4)')
ax.set_xlabel('Component index')
ax.set_ylabel('Singular value')
ax.set_title('Singular Values: Eigendecomp vs Power Iteration')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(np.arange(1, N_COMPONENTS_SVD + 1),
        np.abs(s_eig_cmp - s_pi_cmp), 'g-o', ms=4)
ax.set_xlabel('Component index')
ax.set_ylabel('|s_eig - s_pi|')
ax.set_title('Singular Value Approximation Error')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('svd_algorithm_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nPower iteration is faster for large matrices (only k+oversampling columns of X^T X).')
print('More power iterations => smaller approximation error, diminishing returns after q=4.')

### Singular Value Spectrum & Eckart–Young Theorem

The **scree plot** shows how quickly singular values decay. A sharp "elbow" indicates that most variance lives in a low-dimensional subspace. The **Eckart–Young theorem** guarantees that the rank-$k$ truncation minimises reconstruction error among all rank-$k$ matrices.

In [ ]:
# ── SVD on Digits: singular value spectrum ─────────────────────────────────────
print('Computing full SVD of Digits matrix...')
U_d, s_d, Vt_d = compute_svd_eig(X_digits)
print(f'  U shape: {U_d.shape},  s shape: {s_d.shape},  Vt shape: {Vt_d.shape}')

evr_d: np.ndarray = explained_variance_ratio_svd(s_d)
n90: int = int(np.searchsorted(evr_d, 0.90)) + 1
n95: int = int(np.searchsorted(evr_d, 0.95)) + 1
n99: int = int(np.searchsorted(evr_d, 0.99)) + 1
print(f'  Components for 90% variance: {n90}')
print(f'  Components for 95% variance: {n95}')
print(f'  Components for 99% variance: {n99}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Scree plot — singular values
ax = axes[0]
ax.plot(np.arange(1, 41), s_d[:40], 'b-o', ms=4)
ax.set_xlabel('Component index')
ax.set_ylabel('Singular value')
ax.set_title('Singular Value Spectrum (top 40)')
ax.grid(True, alpha=0.3)

# Cumulative explained variance
ax = axes[1]
ax.plot(np.arange(1, len(evr_d) + 1), evr_d, 'g-')
ax.axhline(0.90, color='orange', ls='--', label=f'90% at k={n90}')
ax.axhline(0.95, color='red', ls='--', label=f'95% at k={n95}')
ax.axhline(0.99, color='purple', ls='--', label=f'99% at k={n99}')
ax.set_xlabel('Number of components k')
ax.set_ylabel('Cumulative explained variance')
ax.set_title('Cumulative Explained Variance')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Eckart-Young: reconstruction error vs rank
ax = axes[2]
ks_ey: list = [1, 2, 4, 8, 16, 24, 32, 40, 48, 56, 64]
errs_ey: list = []
for k_ey in ks_ey:
    X_hat_ey = reconstruct_from_svd(U_d[:, :k_ey], s_d[:k_ey], Vt_d[:k_ey, :])
    errs_ey.append(frobenius_error(X_digits, X_hat_ey))

ax.plot(ks_ey, errs_ey, 'r-o', ms=5)
ax.axvline(N_COMPONENTS_SVD, color='b', ls='--', label=f'k={N_COMPONENTS_SVD} chosen')
ax.set_xlabel('Rank k')
ax.set_ylabel('Relative Frobenius error')
ax.set_title('Eckart-Young: Error vs Rank')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('svd_spectrum.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\nEckart-Young: rank {N_COMPONENTS_SVD} gives rel. error = {errs_ey[ks_ey.index(N_COMPONENTS_SVD)]:.4f}')

### Non-negative Matrix Factorization — Lee–Seung Algorithm

The multiplicative update rules guarantee non-negativity at every step (product of non-negative numbers stays non-negative) and monotonically decrease the Frobenius reconstruction error. Convergence is to a local minimum; the global minimum is generally NP-hard to find.

**Initialisation:** We use uniform random $W, H$ scaled so that $WH$ approximates the mean of $X$. Better initialisations (NNDSVD — Non-negative Double SVD) are available but require the SVD itself as a sub-step.

In [ ]:
# ── NMF multiplicative update (Lee & Seung, 1999) ────────────────────────────

def nmf_multiplicative_update(
    X: np.ndarray,
    W: np.ndarray,
    H: np.ndarray,
    eps: float = NMF_EPS,
) -> tuple:
    '''One step of the Lee-Seung multiplicative update for Frobenius-NMF.

    Update rules:
        H <- H * (W^T X) / (W^T W H + eps)
        W <- W * (X H^T) / (W H H^T + eps)

    Args:
        X: Non-negative target matrix of shape (m, n).
        W: Current basis matrix of shape (m, k), non-negative.
        H: Current coefficient matrix of shape (k, n), non-negative.
        eps: Numerical floor to prevent division by zero.

    Returns:
        Updated tuple (W, H), both non-negative.
    '''
    # ── Update H first (standard order) ──────────────────────────────────────
    WtX: np.ndarray = W.T @ X          # (k, n)
    WtWH: np.ndarray = W.T @ W @ H     # (k, n)
    H = H * (WtX / (WtWH + eps))
    H = np.maximum(H, eps)             # keep non-negative
    # ── Update W ─────────────────────────────────────────────────────────────
    XHt: np.ndarray = X @ H.T          # (m, k)
    WHHt: np.ndarray = W @ (H @ H.T)   # (m, k)
    W = W * (XHt / (WHHt + eps))
    W = np.maximum(W, eps)
    return W, H


def fit_nmf(
    X: np.ndarray,
    n_components: int,
    n_iter: int = NMF_MAX_ITER,
    tol: float = NMF_TOL,
    random_state: int = SEED,
    eps: float = NMF_EPS,
    verbose: bool = False,
) -> dict:
    '''Fit NMF via Lee-Seung multiplicative update rules.

    Args:
        X: Non-negative target matrix of shape (m, n).
        n_components: Factorisation rank k.
        n_iter: Maximum number of multiplicative update iterations.
        tol: Stop when relative change in Frobenius error falls below tol.
        random_state: RNG seed for W, H initialisation.
        eps: Numerical floor for updates.
        verbose: Print progress every 50 iterations.

    Returns:
        Dict with keys 'W', 'H', 'errors' (per-iteration), 'n_iter'.
    '''
    rng = np.random.RandomState(random_state)
    m, n = X.shape
    # Scaled random initialisation
    scale: float = float(np.sqrt(X.mean() / n_components))
    W: np.ndarray = rng.rand(m, n_components) * scale + eps
    H: np.ndarray = rng.rand(n_components, n) * scale + eps
    errors: list = []
    prev_err: float = np.inf
    actual_iter: int = 0
    for it in range(n_iter):
        W, H = nmf_multiplicative_update(X, W, H, eps=eps)
        err: float = float(np.linalg.norm(X - W @ H, 'fro'))
        errors.append(err)
        rel_change: float = abs(prev_err - err) / (prev_err + 1e-10)
        if verbose and (it + 1) % 50 == 0:
            print(f'  Iter {it + 1:4d} | Frob error = {err:.4f} | rel_change = {rel_change:.2e}')
        if rel_change < tol and it > 5:
            actual_iter = it + 1
            if verbose:
                print(f'  Converged at iteration {actual_iter} (rel_change={rel_change:.2e})')
            break
        prev_err = err
        actual_iter = it + 1
    return {'W': W, 'H': H, 'errors': errors, 'n_iter': actual_iter}


# ── NMF verification on tiny matrix ──────────────────────────────────────────
X_nmf_toy = np.array([[4., 2., 0.], [0., 3., 1.], [2., 1., 4.]], dtype=np.float64)
result_nmf_toy = fit_nmf(X_nmf_toy, n_components=2, n_iter=500, tol=1e-6)
W_toy, H_toy = result_nmf_toy['W'], result_nmf_toy['H']
X_nmf_rec = W_toy @ H_toy

print('NMF verification on 3x3 non-negative matrix:')
print(f'  Converged in {result_nmf_toy["n_iter"]} iterations')
print(f'  Reconstruction error: {frobenius_error(X_nmf_toy, X_nmf_rec):.4f}')
print(f'  W non-negative: {bool(np.all(W_toy >= 0))}')
print(f'  H non-negative: {bool(np.all(H_toy >= 0))}')

In [ ]:
# ── NMF convergence curves at different ranks ─────────────────────────────────
print('=== NMF Convergence Curves by Rank ===\n')

K_CONV_LIST: list = [4, 8, 12, 16, 24]
convergence_data: dict = {}

for k_conv in K_CONV_LIST:
    result_conv = fit_nmf(
        X_digits_nn, n_components=k_conv, n_iter=200, tol=1e-7, verbose=False,
    )
    convergence_data[k_conv] = result_conv['errors']
    print(f'  k={k_conv:3d} | iters={result_conv["n_iter"]:3d} | final error={result_conv["errors"][-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors_conv = plt.cm.viridis(np.linspace(0, 1, len(K_CONV_LIST)))

ax = axes[0]
for ci, (k_c, errs_c) in enumerate(convergence_data.items()):
    ax.plot(errs_c, color=colors_conv[ci], lw=2, label=f'k={k_c}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Frobenius error')
ax.set_title('NMF Convergence by Rank (linear scale)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
for ci, (k_c, errs_c) in enumerate(convergence_data.items()):
    ax.semilogy(errs_c, color=colors_conv[ci], lw=2, label=f'k={k_c}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Frobenius error (log scale)')
ax.set_title('NMF Convergence by Rank (log scale)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('nmf_convergence_curves.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nKey: multiplicative updates are monotone (error never increases).')
print('Higher rank => larger initial absolute error but potentially lower asymptote.')

### Matrix Completion via Iterative SVD

When only a fraction of matrix entries are observed (e.g., a user has only rated some movies), we need to **impute missing values** under a low-rank assumption. The iterative SVD algorithm alternates between:
1. **Imputing** missing entries with the current low-rank estimate.
2. **Projecting** onto rank-$k$ matrices via truncated SVD.

This is a simplified version of the **SoftImpute** algorithm and the approach taken in the Netflix Prize winning solution.

In [ ]:
def matrix_completion_svd(
    X_obs: np.ndarray,
    mask: np.ndarray,
    k: int,
    n_iter: int = 60,
    tol: float = 1e-4,
    verbose: bool = True,
) -> np.ndarray:
    '''Recover a low-rank matrix from partial observations using iterative SVD.

    Algorithm (hard-threshold iterative SVD / simplified SoftImpute):
      1. Fill missing entries with column means.
      2. Repeat:
         a. Compute rank-k truncated SVD of the current filled matrix.
         b. Reconstruct: X_hat = U_k Sigma_k Vt_k.
         c. Update only missing positions: X_fill[missing] <- X_hat[missing].
      3. Stop when Frobenius change in X_fill < tol.

    Args:
        X_obs: Observed matrix (m, n) with NaN at missing positions.
        mask: Boolean mask (m, n), True = entry is observed.
        k: Target rank for the SVD step.
        n_iter: Maximum iterations.
        tol: Convergence tolerance on the Frobenius norm of the change.
        verbose: Print convergence info.

    Returns:
        Completed matrix of shape (m, n).
    '''
    X_fill: np.ndarray = X_obs.copy()
    # Initialise missing entries with column means of observed values
    col_means: np.ndarray = np.nanmean(X_obs, axis=0)
    for j in range(X_obs.shape[1]):
        X_fill[~mask[:, j], j] = col_means[j]

    prev_fill: np.ndarray = X_fill.copy()
    for it in range(n_iter):
        U_k, s_k, Vt_k = truncated_svd_manual(X_fill, k)
        X_hat: np.ndarray = reconstruct_from_svd(U_k, s_k, Vt_k)
        X_new: np.ndarray = X_fill.copy()
        X_new[~mask] = X_hat[~mask]                  # update only missing
        change: float = float(np.linalg.norm(X_new - prev_fill, 'fro'))
        X_fill = X_new
        prev_fill = X_fill.copy()
        if verbose and (it + 1) % 10 == 0:
            print(f'  Iter {it + 1:3d} | change = {change:.4e}')
        if change < tol:
            if verbose:
                print(f'  Converged at iteration {it + 1}')
            break
    return X_fill

### Low-Rank Decomposition & the LoRA Connection

**LoRA** (Hu et al., 2021) parametrises weight updates as $\Delta W = BA$ where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$, with $r \ll \min(d, k)$. The SVD provides the *optimal* such initialisation: $B = U_r \sqrt{\Sigma_r}$, $A = \sqrt{\Sigma_r} V_r^\top$ minimises the approximation error among all rank-$r$ factorisations. In practice LoRA initialises $A$ from a Gaussian and $B$ as zero so that $\Delta W = 0$ at the start of fine-tuning, but the theoretical connection to SVD motivates the design.

**Connection to recommender systems (Module 19):** User–item rating matrices are factored into $U_\text{user} \in \mathbb{R}^{|\text{users}| \times k}$ and $V_\text{item} \in \mathbb{R}^{|\text{items}| \times k}$, exactly the SVD structure but learned via gradient descent on observed ratings rather than closed-form eigendecomposition.

In [ ]:
def lora_decompose(W: np.ndarray, rank: int) -> tuple:
    '''Decompose weight matrix W into low-rank factors B, A via SVD.

    Finds B (d x rank) and A (rank x k) such that B @ A is the best
    rank-r approximation to W (Eckart-Young theorem).

    Args:
        W: Weight matrix of shape (d, k).
        rank: LoRA rank r.

    Returns:
        Tuple (B, A) where B has shape (d, rank) and A has shape (rank, k).
    '''
    U_r, s_r, Vt_r = truncated_svd_manual(W, rank)
    sqrt_s: np.ndarray = np.sqrt(s_r)
    B: np.ndarray = U_r * sqrt_s[np.newaxis, :]    # (d, r)
    A: np.ndarray = sqrt_s[:, np.newaxis] * Vt_r   # (r, k)
    return B, A


# ── LoRA demo: decompose a random weight matrix ────────────────────────────────
rng_lora = np.random.RandomState(SEED)
W_demo: np.ndarray = rng_lora.randn(WEIGHT_DIM_OUT, WEIGHT_DIM_IN)  # (64, 128)

print(f'Weight matrix W: {W_demo.shape}')
print(f'LoRA rank r = {LORA_RANK}  (parameter reduction: {LORA_RANK*(WEIGHT_DIM_OUT+WEIGHT_DIM_IN)} vs {WEIGHT_DIM_OUT*WEIGHT_DIM_IN})')

B_lora, A_lora = lora_decompose(W_demo, LORA_RANK)
W_approx: np.ndarray = B_lora @ A_lora

print(f'B shape: {B_lora.shape},  A shape: {A_lora.shape}')
print(f'Reconstruction error (rank {LORA_RANK}): {frobenius_error(W_demo, W_approx):.4f}')

# Sweep ranks to show approximation quality
lora_ranks: list = [1, 2, 4, 8, 16, 32, 48, 64]
lora_errors: list = []
lora_params: list = []
full_params: int = WEIGHT_DIM_OUT * WEIGHT_DIM_IN

for r_l in lora_ranks:
    B_l, A_l = lora_decompose(W_demo, r_l)
    err_l = frobenius_error(W_demo, B_l @ A_l)
    n_params_l = r_l * (WEIGHT_DIM_OUT + WEIGHT_DIM_IN)
    lora_errors.append(err_l)
    lora_params.append(n_params_l)
    ratio_l = n_params_l / full_params
    print(f'  rank={r_l:3d} | error={err_l:.4f} | params={n_params_l:5d} ({ratio_l:.1%} of full)')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.plot(lora_ranks, lora_errors, 'b-o', ms=6)
ax.axvline(LORA_RANK, color='r', ls='--', label=f'r={LORA_RANK}')
ax.set_xlabel('LoRA rank r')
ax.set_ylabel('Relative Frobenius error')
ax.set_title('LoRA Approximation Error vs Rank')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
param_pcts = [100 * p / full_params for p in lora_params]
ax.plot(param_pcts, lora_errors, 'g-s', ms=6)
ax.set_xlabel('Parameter budget (% of full matrix)')
ax.set_ylabel('Relative Frobenius error')
ax.set_title('Error vs Parameter Budget')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lora_error_vs_rank.png', dpi=100, bbox_inches='tight')
plt.show()

---
## Part 2 — Putting It All Together

We now assemble the building blocks into sklearn-style classes: `TruncatedSVD` and `NMF`, each with `fit`, `transform`, and `fit_transform` methods.

In [ ]:
class TruncatedSVD:
    '''Dimensionality reduction via truncated Singular Value Decomposition.

    Computes the top-k left singular vectors of X and projects data
    into that k-dimensional subspace. Analogous to PCA but works
    directly on the data matrix (no centering required, though centering
    is recommended for PCA-style use).

    Args:
        n_components: Number of singular components to retain.

    Attributes:
        components_: Right singular vectors Vt, shape (n_components, n_features).
        singular_values_: Top singular values, shape (n_components,).
        explained_variance_ratio_: Per-component explained variance ratio.
        cumulative_variance_ratio_: Cumulative explained variance ratio.
    '''

    def __init__(self, n_components: int = 2) -> None:
        '''Initialise TruncatedSVD.

        Args:
            n_components: Number of singular components to retain.
        '''
        self.n_components = n_components
        self.components_: np.ndarray | None = None
        self.singular_values_: np.ndarray | None = None
        self.explained_variance_ratio_: np.ndarray | None = None
        self.cumulative_variance_ratio_: np.ndarray | None = None
        self._s_full: np.ndarray | None = None

    def fit(self, X: np.ndarray) -> 'TruncatedSVD':
        '''Fit the model: compute the truncated SVD of X.

        Args:
            X: Data matrix of shape (n_samples, n_features).

        Returns:
            Self (for chaining).
        '''
        U, s, Vt = compute_svd_eig(X)
        self._s_full = s
        k: int = min(self.n_components, len(s))
        self.components_ = Vt[:k, :]           # (k, n_features)
        self.singular_values_ = s[:k]           # (k,)
        all_evr = explained_variance_ratio_svd(s)
        self.explained_variance_ratio_ = np.diff(
            np.concatenate([[0.0], all_evr[:k]])
        )
        self.cumulative_variance_ratio_ = all_evr[:k]
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        '''Project X onto the top-k right singular vectors.

        Args:
            X: Data matrix (n_samples, n_features).

        Returns:
            Projected data of shape (n_samples, n_components).
        '''
        if self.components_ is None:
            raise RuntimeError('Call fit() before transform().')
        return X @ self.components_.T  # (n_samples, k)

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        '''Fit and return projection of X.

        Args:
            X: Data matrix (n_samples, n_features).

        Returns:
            Projected data of shape (n_samples, n_components).
        '''
        return self.fit(X).transform(X)

    def inverse_transform(self, Z: np.ndarray) -> np.ndarray:
        '''Map from reduced space back to original space.

        Args:
            Z: Reduced data (n_samples, n_components).

        Returns:
            Reconstructed matrix (n_samples, n_features).
        '''
        if self.components_ is None:
            raise RuntimeError('Call fit() before inverse_transform().')
        return Z @ self.components_  # (n_samples, n_features)

In [ ]:
class NMF:
    '''Non-negative Matrix Factorization: X approx W H, with W, H >= 0.

    Fitted via Lee-Seung multiplicative update rules on the Frobenius
    reconstruction objective. Yields parts-based representations where
    each component is a non-negative basis element.

    Args:
        n_components: Factorisation rank k.
        n_iter: Maximum multiplicative update iterations.
        tol: Stop when relative error change falls below tol.
        random_state: Seed for weight initialisation.
        eps: Numerical floor.
        verbose: Print convergence info during fit.

    Attributes:
        components_: Basis matrix H of shape (n_components, n_features).
        n_iter_: Actual iterations run.
        reconstruction_error_: Final Frobenius error.
    '''

    def __init__(
        self,
        n_components: int = 10,
        n_iter: int = NMF_MAX_ITER,
        tol: float = NMF_TOL,
        random_state: int = SEED,
        eps: float = NMF_EPS,
        verbose: bool = False,
    ) -> None:
        '''Initialise NMF.

        Args:
            n_components: Factorisation rank k.
            n_iter: Maximum multiplicative update iterations.
            tol: Stop when relative error change falls below tol.
            random_state: Seed for W, H weight initialisation.
            eps: Numerical floor to prevent division by zero.
            verbose: Print convergence progress during fit.
        '''
        self.n_components = n_components
        self.n_iter = n_iter
        self.tol = tol
        self.random_state = random_state
        self.eps = eps
        self.verbose = verbose
        self.components_: np.ndarray | None = None
        self._W: np.ndarray | None = None
        self.n_iter_: int = 0
        self.reconstruction_error_: float = np.inf
        self._fit_errors: list = []

    def fit(self, X: np.ndarray) -> 'NMF':
        '''Fit NMF on X. X must be non-negative.

        Args:
            X: Non-negative data matrix (n_samples, n_features).

        Returns:
            Self (for chaining).
        '''
        result = fit_nmf(
            X, self.n_components, self.n_iter, self.tol,
            self.random_state, self.eps, self.verbose,
        )
        self._W = result['W']
        self.components_ = result['H']       # (k, n_features)
        self.n_iter_ = result['n_iter']
        self._fit_errors = result['errors']
        self.reconstruction_error_ = result['errors'][-1] if result['errors'] else np.inf
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        '''Compute coefficient matrix W for new data X (H fixed from fit).

        For held-out data, we solve min_{W>=0} ||X - W H||_F^2 by running
        a short multiplicative update loop on W only.

        Args:
            X: Non-negative data matrix (n_samples, n_features).

        Returns:
            Coefficient matrix W of shape (n_samples, n_components).
        '''
        if self.components_ is None:
            raise RuntimeError('Call fit() before transform().')
        H = self.components_
        rng = np.random.RandomState(self.random_state)
        scale: float = float(np.sqrt(X.mean() / self.n_components))
        W = rng.rand(X.shape[0], self.n_components) * scale + self.eps
        for _ in range(100):
            XHt = X @ H.T
            WHHt = W @ (H @ H.T)
            W = W * (XHt / (WHHt + self.eps))
            W = np.maximum(W, self.eps)
        return W

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        '''Fit NMF and return W (coefficient matrix for training data).

        Args:
            X: Non-negative data matrix (n_samples, n_features).

        Returns:
            Coefficient matrix W of shape (n_samples, n_components).
        '''
        self.fit(X)
        return self._W  # type: ignore[return-value]

    def inverse_transform(self, W: np.ndarray) -> np.ndarray:
        '''Reconstruct X from coefficient matrix W: X_hat = W H.

        Args:
            W: Coefficient matrix (n_samples, n_components).

        Returns:
            Reconstructed matrix (n_samples, n_features).
        '''
        if self.components_ is None:
            raise RuntimeError('Call fit() before inverse_transform().')
        return W @ self.components_

    def reconstruction_error(self, X: np.ndarray) -> float:
        '''Frobenius reconstruction error on new data X.

        Args:
            X: Non-negative data matrix.

        Returns:
            Absolute Frobenius norm of X - W H.
        '''
        W = self.transform(X)
        return float(np.linalg.norm(X - W @ self.components_, 'fro'))

### Sanity Check on Toy Data

In [ ]:
# ── TruncatedSVD sanity check ─────────────────────────────────────────────────
rng_san = np.random.RandomState(SEED)
X_san = rng_san.randn(50, 20)   # 50 samples, 20 features

tsvd_san = TruncatedSVD(n_components=5)
Z_san = tsvd_san.fit_transform(X_san)
X_rec_san = tsvd_san.inverse_transform(Z_san)

print('TruncatedSVD sanity check (50x20 matrix, k=5):')
print(f'  Projection shape: {Z_san.shape}')
print(f'  Cum var ratio (5 comps): {tsvd_san.cumulative_variance_ratio_[-1]:.3f}')
print(f'  Reconstruction error: {frobenius_error(X_san, X_rec_san):.4f}')

# Verify orthonormality of V components
VVt = tsvd_san.components_ @ tsvd_san.components_.T
orth_err = float(np.max(np.abs(VVt - np.eye(5))))
print(f'  Orthonormality error (V V^T - I): {orth_err:.2e}')

# ── NMF sanity check ──────────────────────────────────────────────────────────
X_san_nn = np.abs(rng_san.randn(30, 15))  # non-negative
nmf_san = NMF(n_components=3, n_iter=200, verbose=False)
W_san = nmf_san.fit_transform(X_san_nn)
X_rec_nmf = nmf_san.inverse_transform(W_san)

print('\nNMF sanity check (30x15 non-negative matrix, k=3):')
print(f'  W shape: {W_san.shape},  H shape: {nmf_san.components_.shape}')
print(f'  Converged in {nmf_san.n_iter_} iterations')
print(f'  W non-negative: {bool(np.all(W_san >= 0))}')
print(f'  H non-negative: {bool(np.all(nmf_san.components_ >= 0))}')
print(f'  Reconstruction error: {frobenius_error(X_san_nn, X_rec_nmf):.4f}')

---
## Part 3 — Application to Digits & Matrix Completion

In [ ]:
# ── Truncated SVD on Digits ───────────────────────────────────────────────────
print(f'Applying TruncatedSVD (k={N_COMPONENTS_SVD}) to Digits...')
tsvd_digits = TruncatedSVD(n_components=N_COMPONENTS_SVD)
Z_digits: np.ndarray = tsvd_digits.fit_transform(X_digits)

print(f'  Projection: {X_digits.shape} -> {Z_digits.shape}')
cum_var = float(tsvd_digits.cumulative_variance_ratio_[-1])
print(f'  Cumulative variance explained: {cum_var:.3f}')

# 2D projection for visualization (top 2 components)
tsvd_2d = TruncatedSVD(n_components=2)
Z_2d: np.ndarray = tsvd_2d.fit_transform(X_digits)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
colors = plt.cm.tab10(np.linspace(0, 1, 10))
for digit_cls in range(10):
    mask_cls = y_digits == digit_cls
    ax.scatter(Z_2d[mask_cls, 0], Z_2d[mask_cls, 1],
               c=[colors[digit_cls]], s=10, alpha=0.5, label=str(digit_cls))
ax.set_xlabel('SVD Component 1')
ax.set_ylabel('SVD Component 2')
ax.set_title('Digits Projected onto Top-2 SVD Components')
ax.legend(title='Digit', fontsize=8, markerscale=2, ncol=2)
ax.grid(True, alpha=0.3)

# Reconstructed images at various ranks
ax = axes[1]
sample_idx = np.where(y_digits == 4)[0][0]
X_sample = X_digits[[sample_idx], :]  # (1, 64)
ranks_vis = [1, 4, 8, 16, 32, 64]
n_vis = len(ranks_vis)
imgs_to_show = []
for rv in ranks_vis:
    tsvd_rv = TruncatedSVD(n_components=rv)
    tsvd_rv.fit(X_digits)
    Z_rv = tsvd_rv.transform(X_sample)
    rec_rv = tsvd_rv.inverse_transform(Z_rv).clip(0, 16).reshape(8, 8)
    imgs_to_show.append(rec_rv)

combined = np.concatenate(imgs_to_show, axis=1)
ax.imshow(combined, cmap='gray_r', vmin=0, vmax=16)
ax.axis('off')
label_positions = [4 + 8 * i for i in range(n_vis)]
ax.set_xticks(label_positions)
ax.set_xticklabels([f'k={r}' for r in ranks_vis], fontsize=9)
ax.set_title(f'Digit "{y_digits[sample_idx]}" Reconstructed at Different Ranks')

plt.tight_layout()
plt.savefig('svd_digits_projection.png', dpi=100, bbox_inches='tight')
plt.show()

### NMF Basis Components — Parts-Based Representation

Unlike SVD components (which can have negative values and represent global correlations), NMF components are non-negative and tend to correspond to **local parts** of the image — strokes, corners, loops. Each component row of $H$ is a basis "stroke" and each column of $W$ is the non-negative coefficient indicating how much of each stroke appears in that sample.

In [ ]:
# ── NMF on Digits: parts-based decomposition ──────────────────────────────────
print(f'Fitting NMF (k={N_COMPONENTS_NMF}) on Digits (normalised to [0,1])...')
nmf_digits = NMF(n_components=N_COMPONENTS_NMF, n_iter=NMF_MAX_ITER, verbose=True)
W_digits: np.ndarray = nmf_digits.fit_transform(X_digits_nn)

print(f'\n  W shape: {W_digits.shape}')
print(f'  H (components_) shape: {nmf_digits.components_.shape}')
cum_err = frobenius_error(X_digits_nn, W_digits @ nmf_digits.components_)
print(f'  Relative reconstruction error: {cum_err:.4f}')

# Visualise the 16 NMF basis components as 8x8 images
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for ci in range(N_COMPONENTS_NMF):
    ax = axes[ci // 4, ci % 4]
    component_img = nmf_digits.components_[ci].reshape(8, 8)
    ax.imshow(component_img, cmap='hot')
    ax.axis('off')
    ax.set_title(f'C{ci}', fontsize=8)
plt.suptitle(f'NMF Basis Components (k={N_COMPONENTS_NMF}) — Digit Strokes', fontsize=12)
plt.tight_layout()
plt.savefig('nmf_components.png', dpi=100, bbox_inches='tight')
plt.show()

# Compare a single reconstruction: SVD vs NMF
sample_idx2 = np.where(y_digits == 8)[0][0]
x_orig = X_digits_nn[[sample_idx2], :]  # (1, 64)

tsvd_comp = TruncatedSVD(n_components=N_COMPONENTS_NMF)
tsvd_comp.fit(X_digits_nn)
z_svd = tsvd_comp.transform(x_orig)
x_svd_rec = tsvd_comp.inverse_transform(z_svd).clip(0, 1)

w_nmf = nmf_digits.transform(x_orig)
x_nmf_rec = nmf_digits.inverse_transform(w_nmf).clip(0, 1)

fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax_r, img, title in zip(
    axes,
    [x_orig.reshape(8, 8), x_svd_rec.reshape(8, 8), x_nmf_rec.reshape(8, 8)],
    ['Original', f'SVD (k={N_COMPONENTS_NMF})', f'NMF (k={N_COMPONENTS_NMF})'],
):
    ax_r.imshow(img, cmap='gray_r', vmin=0, vmax=1)
    ax_r.axis('off')
    ax_r.set_title(title)
plt.suptitle(f'Reconstruction Comparison — Digit "{y_digits[sample_idx2]}"')
plt.tight_layout()
plt.savefig('svd_vs_nmf_reconstruction.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Matrix completion on synthetic low-rank data ──────────────────────────────
print(f'Matrix completion on {M_ROWS}x{N_COLS} rank-{TRUE_RANK} matrix')
print(f'  Missing: {MISSING_FRAC:.0%} of entries')
print(f'  Recovery rank: k = {TRUE_RANK}')
print()

X_completed: np.ndarray = matrix_completion_svd(
    X_observed, mask_obs, k=TRUE_RANK, n_iter=60, tol=1e-4, verbose=True,
)

# Evaluate only on the originally missing entries
missing_err = float(np.sqrt(np.mean((X_completed[~mask_obs] - X_true[~mask_obs]) ** 2)))
baseline_err = float(np.sqrt(np.mean((np.nanmean(X_observed) - X_true[~mask_obs]) ** 2)))
print(f'\nResults (evaluated on missing entries only):')
print(f'  RMSE (column-mean baseline) : {baseline_err:.4f}')
print(f'  RMSE (iterative SVD)        : {missing_err:.4f}')
print(f'  Improvement: {(1 - missing_err / baseline_err):.1%}')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
vmin_mc, vmax_mc = X_true.min(), X_true.max()
for ax_mc, mat, title in zip(
    axes,
    [X_true, X_observed, X_completed],
    ['Ground Truth', 'Observed (missing=nan)', 'Completed (iterative SVD)'],
):
    im_mc = ax_mc.imshow(mat, cmap='coolwarm', vmin=vmin_mc, vmax=vmax_mc, aspect='auto')
    ax_mc.set_title(title)
    ax_mc.set_xlabel('Column')
    ax_mc.set_ylabel('Row')
    plt.colorbar(im_mc, ax=ax_mc)
plt.tight_layout()
plt.savefig('matrix_completion.png', dpi=100, bbox_inches='tight')
plt.show()

### Library Comparison

We verify our `TruncatedSVD` and `NMF` against sklearn equivalents and note performance differences.

In [ ]:
# ── TruncatedSVD comparison ───────────────────────────────────────────────────
t0 = time.perf_counter()
tsvd_ours = TruncatedSVD(n_components=N_COMPONENTS_SVD)
Z_ours = tsvd_ours.fit_transform(X_digits)
time_ours_svd = time.perf_counter() - t0

t0 = time.perf_counter()
tsvd_sk = SklearnTSVD(n_components=N_COMPONENTS_SVD, random_state=SEED)
Z_sk = tsvd_sk.fit_transform(X_digits)
time_sk_svd = time.perf_counter() - t0

# Components may differ in sign — compare absolute values
corr_svd: float = float(np.mean([abs(np.corrcoef(Z_ours[:, i], np.abs(Z_sk[:, i]))[0, 1])
                                  for i in range(N_COMPONENTS_SVD)]))

print('TruncatedSVD comparison:')
print(f'  Our time    : {time_ours_svd:.3f}s')
print(f'  Sklearn time: {time_sk_svd:.3f}s')
print(f'  Mean |correlation| between projections: {corr_svd:.4f}')
print(f'  Our cum var ratio   : {float(tsvd_ours.cumulative_variance_ratio_[-1]):.4f}')
print(f'  Sklearn cum var ratio: {float(tsvd_sk.explained_variance_ratio_.sum()):.4f}')

# ── NMF comparison ────────────────────────────────────────────────────────────
t0 = time.perf_counter()
nmf_ours = NMF(n_components=N_COMPONENTS_NMF, n_iter=NMF_MAX_ITER)
W_ours = nmf_ours.fit_transform(X_digits_nn)
time_ours_nmf = time.perf_counter() - t0

t0 = time.perf_counter()
nmf_sk = SklearnNMF(n_components=N_COMPONENTS_NMF, max_iter=NMF_MAX_ITER,
                    random_state=SEED, init='random')
W_sk_nmf = nmf_sk.fit_transform(X_digits_nn)
time_sk_nmf = time.perf_counter() - t0

err_ours_nmf = frobenius_error(X_digits_nn, W_ours @ nmf_ours.components_)
err_sk_nmf = frobenius_error(X_digits_nn, W_sk_nmf @ nmf_sk.components_)

print('\nNMF comparison:')
print(f'  Our time    : {time_ours_nmf:.3f}s  | rel error: {err_ours_nmf:.4f}')
print(f'  Sklearn time: {time_sk_nmf:.3f}s  | rel error: {err_sk_nmf:.4f}')
print('  Note: sklearn uses optimised LAPACK; our implementation is pure NumPy.')

---
## Part 4 — Evaluation & Analysis

In [ ]:
# ── Ablation 1: Reconstruction error vs rank (SVD and NMF) ────────────────────
print('=== Rank Ablation: Reconstruction Error vs k ===\n')

ranks_abl: list = [1, 2, 4, 6, 8, 12, 16, 20, 24, 32, 40, 48, 56, 64]
svd_errors: list = []
nmf_errors: list = []

for k_abl in ranks_abl:
    # SVD error
    tsvd_abl = TruncatedSVD(n_components=k_abl)
    Z_abl = tsvd_abl.fit_transform(X_digits)
    X_svd_rec = tsvd_abl.inverse_transform(Z_abl)
    svd_err = frobenius_error(X_digits, X_svd_rec)
    svd_errors.append(svd_err)

    # NMF error (use fewer iterations for speed in ablation)
    nmf_abl = NMF(n_components=k_abl, n_iter=150, verbose=False)
    W_abl = nmf_abl.fit_transform(X_digits_nn)
    nmf_err = frobenius_error(X_digits_nn, W_abl @ nmf_abl.components_)
    nmf_errors.append(nmf_err)

    print(f'  k={k_abl:3d} | SVD err={svd_err:.4f} | NMF err={nmf_err:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(ranks_abl, svd_errors, 'b-o', ms=5, label='SVD (ours)')
ax.plot(ranks_abl, nmf_errors, 'r-s', ms=5, label='NMF (ours)')
ax.axvline(N_COMPONENTS_SVD, color='gray', ls='--', alpha=0.7, label=f'k={N_COMPONENTS_SVD}')
ax.set_xlabel('Rank k')
ax.set_ylabel('Relative Frobenius error')
ax.set_title('Reconstruction Error vs Rank')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.semilogy(ranks_abl, svd_errors, 'b-o', ms=5, label='SVD')
ax.semilogy(ranks_abl, nmf_errors, 'r-s', ms=5, label='NMF')
ax.set_xlabel('Rank k')
ax.set_ylabel('Relative error (log scale)')
ax.set_title('Reconstruction Error (log scale)')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('rank_ablation.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nObservation: SVD achieves the theoretical minimum error at each rank')
print('(Eckart-Young). NMF is always >= SVD error due to non-negativity constraint.')

In [ ]:
# ── NMF component interpretability analysis ───────────────────────────────────
print('=== NMF Component Interpretability Analysis ===\n')

# Which NMF component is most activated for each digit class?
activation_table: np.ndarray = np.zeros((10, N_COMPONENTS_NMF))
for cls in range(10):
    mask_cls = y_digits == cls
    W_cls = W_digits[mask_cls, :]
    activation_table[cls, :] = W_cls.mean(axis=0)

print('Mean NMF activation per digit class (rows=digit, cols=component):')
df_act = pd.DataFrame(
    activation_table,
    index=[f'Digit {c}' for c in range(10)],
    columns=[f'C{i}' for i in range(N_COMPONENTS_NMF)],
)
print(df_act.round(3).to_string())

# Dominant component per digit
print('\nDominant NMF component per digit class:')
for cls in range(10):
    dom_comp = int(np.argmax(activation_table[cls, :]))
    dom_val = float(activation_table[cls, dom_comp])
    print(f'  Digit {cls}: Component C{dom_comp} (mean activation={dom_val:.3f})')

# Heatmap
fig, ax = plt.subplots(figsize=(12, 4))
im_act = ax.imshow(activation_table, cmap='YlOrRd', aspect='auto')
ax.set_yticks(range(10))
ax.set_yticklabels([str(c) for c in range(10)])
ax.set_xticks(range(N_COMPONENTS_NMF))
ax.set_xticklabels([f'C{i}' for i in range(N_COMPONENTS_NMF)], fontsize=8)
ax.set_xlabel('NMF Component')
ax.set_ylabel('Digit Class')
ax.set_title('Mean NMF Activation per Digit Class')
plt.colorbar(im_act, ax=ax, label='Mean activation')
plt.tight_layout()
plt.savefig('nmf_activation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nObservation: Different digit classes activate different NMF components,')
print('reflecting the parts-based structure (strokes, curves) of each digit.')

In [ ]:
# ── Per-class reconstruction error: SVD vs NMF ────────────────────────────────
print('=== Per-Class Reconstruction Error: SVD vs NMF ===\n')

tsvd_cmp = TruncatedSVD(n_components=N_COMPONENTS_NMF)
tsvd_cmp.fit(X_digits_nn)

per_class_svd: list = []
per_class_nmf: list = []

for cls in range(10):
    mask_cls = y_digits == cls
    X_cls = X_digits_nn[mask_cls, :]

    # SVD reconstruction
    Z_cls_svd = tsvd_cmp.transform(X_cls)
    X_cls_svd_rec = tsvd_cmp.inverse_transform(Z_cls_svd).clip(0, 1)
    err_svd_cls = frobenius_error(X_cls, X_cls_svd_rec)
    per_class_svd.append(err_svd_cls)

    # NMF reconstruction
    W_cls_nmf = nmf_digits.transform(X_cls)
    X_cls_nmf_rec = nmf_digits.inverse_transform(W_cls_nmf).clip(0, 1)
    err_nmf_cls = frobenius_error(X_cls, X_cls_nmf_rec)
    per_class_nmf.append(err_nmf_cls)

    diff = err_nmf_cls - err_svd_cls
    print(f'  Digit {cls}: SVD={err_svd_cls:.4f} | NMF={err_nmf_cls:.4f} | diff={diff:+.4f}')

print(f'\n  Mean SVD error: {np.mean(per_class_svd):.4f}')
print(f'  Mean NMF error: {np.mean(per_class_nmf):.4f}')
print(f'  NMF overhead  : {np.mean(per_class_nmf) - np.mean(per_class_svd):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
x_cls_pos = np.arange(10)
width = 0.38
ax.bar(x_cls_pos - width / 2, per_class_svd, width=width,
       label=f'SVD (k={N_COMPONENTS_NMF})', color='steelblue', alpha=0.85)
ax.bar(x_cls_pos + width / 2, per_class_nmf, width=width,
       label=f'NMF (k={N_COMPONENTS_NMF})', color='coral', alpha=0.85)
ax.set_xlabel('Digit class')
ax.set_ylabel('Relative Frobenius error')
ax.set_title('Per-Class Reconstruction Error: SVD vs NMF')
ax.set_xticks(x_cls_pos)
ax.set_xticklabels([str(c) for c in range(10)])
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

ax = axes[1]
diffs = [nmf - svd for nmf, svd in zip(per_class_nmf, per_class_svd)]
colors_diff = ['red' if d > 0 else 'green' for d in diffs]
ax.bar(x_cls_pos, diffs, color=colors_diff, alpha=0.8)
ax.axhline(0, color='black', lw=1)
ax.set_xlabel('Digit class')
ax.set_ylabel('NMF error - SVD error')
ax.set_title('Extra Reconstruction Cost of NMF vs SVD')
ax.set_xticks(x_cls_pos)
ax.set_xticklabels([str(c) for c in range(10)])
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('per_class_reconstruction.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nSVD always achieves lower or equal error (Eckart-Young guarantee).')
print('Some digit classes are harder to reconstruct non-negatively due to complex strokes.')

In [ ]:
# ── Ablation 2: Matrix completion — rank and missing fraction sweep ────────────
print('=== Matrix Completion: Recovery Rank vs RMSE ===\n')

recovery_ranks: list = [1, 2, 3, 4, 5, 6, 8, 10]
rmse_by_rank: list = []

for k_mc in recovery_ranks:
    X_comp_k = matrix_completion_svd(
        X_observed, mask_obs, k=k_mc, n_iter=40, tol=1e-4, verbose=False,
    )
    rmse_k = float(np.sqrt(np.mean((X_comp_k[~mask_obs] - X_true[~mask_obs]) ** 2)))
    rmse_by_rank.append(rmse_k)
    print(f'  k={k_mc:2d} | RMSE (missing) = {rmse_k:.4f}')

best_k_mc = recovery_ranks[int(np.argmin(rmse_by_rank))]
print(f'\nBest recovery rank: {best_k_mc} (true rank: {TRUE_RANK})')

# Missing fraction sweep: how robust is completion as more data is hidden?
print('\n--- Missing Fraction Sweep ---')
frac_list: list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
rmse_by_frac: list = []

rng_frac = np.random.RandomState(SEED + 1)
for frac in frac_list:
    mask_frac = rng_frac.rand(M_ROWS, N_COLS) > frac
    X_obs_frac = X_true.copy()
    X_obs_frac[~mask_frac] = np.nan
    X_comp_frac = matrix_completion_svd(
        X_obs_frac, mask_frac, k=TRUE_RANK, n_iter=40, tol=1e-4, verbose=False,
    )
    if (~mask_frac).sum() > 0:
        rmse_frac = float(np.sqrt(np.mean((X_comp_frac[~mask_frac] - X_true[~mask_frac]) ** 2)))
    else:
        rmse_frac = 0.0
    rmse_by_frac.append(rmse_frac)
    print(f'  missing={frac:.0%} | RMSE={rmse_frac:.4f} | observed={mask_frac.mean():.1%}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(recovery_ranks, rmse_by_rank, 'b-o', ms=6)
ax.axvline(TRUE_RANK, color='r', ls='--', label=f'True rank = {TRUE_RANK}')
ax.set_xlabel('Recovery rank k')
ax.set_ylabel('RMSE on missing entries')
ax.set_title('Matrix Completion: Rank vs RMSE')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot([f * 100 for f in frac_list], rmse_by_frac, 'g-s', ms=6)
ax.axvline(MISSING_FRAC * 100, color='r', ls='--', label=f'Default {MISSING_FRAC:.0%}')
ax.set_xlabel('Missing fraction (%)')
ax.set_ylabel('RMSE on missing entries')
ax.set_title('Matrix Completion: Missing Fraction vs RMSE')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('completion_rank_fraction_ablation.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nOver-estimating rank picks up noise; under-estimating loses signal.')
print('Recovery degrades gracefully as missing fraction increases; fails beyond ~70%.')

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **SVD is the gold standard for low-rank approximation.** The Eckart–Young theorem guarantees that the rank-$k$ SVD truncation minimises Frobenius reconstruction error among all rank-$k$ matrices — no other factorisation can do better.

2. **Non-negativity yields interpretability.** NMF's constraint $W, H \geq 0$ forces a parts-based decomposition where components represent local, additive structures (digit strokes) rather than the holistic, cancelling patterns of SVD components.

3. **Matrix completion is feasible under the low-rank assumption.** Iterative SVD accurately recovers missing entries when the true data matrix is approximately low-rank — the key assumption behind recommender systems and collaborative filtering.

4. **LoRA is a practitioner's application of SVD.** Representing a weight update $\Delta W \approx BA$ with $r \ll d$ exploits the empirical observation that neural network weight updates are approximately low-rank, reducing trainable parameters by orders of magnitude.

5. **NMF error is always $\geq$ SVD error** at the same rank, because NMF is solving a constrained version of the same optimisation problem — the non-negativity constraint restricts the feasible set.

### What's Next

→ **03-10 (Bayesian Inference & Probabilistic Thinking)** extends the probabilistic view of learning — where SVD and NMF find point estimates, Bayesian methods provide full posterior distributions over parameters and predictions.

### Going Further

- Lee & Seung (1999): *Learning the Parts of Objects by Non-negative Matrix Factorization* — the original NMF paper.
- Mazumder, Hastie & Tibshirani (2010): *Spectral Regularization Algorithms for Learning Large Incomplete Matrices* — SoftImpute for matrix completion.
- Hu et al. (2021): *LoRA: Low-Rank Adaptation of Large Language Models* — the LoRA paper.